# 31B Task suite, release morning and economics
Grade the agent by what it did, read from the trace and the ledger, then run one release morning end to end and price it.

In [ ]:
import sys, json, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "creditwatch").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, yaml
from creditwatch import (approvals, audit, catalog, channel, clock, econ, evals, heartbeat, llm, loop,
                         memory, panel, policy, signals, skills, standin, workspace)
P = yaml.safe_load(open(ROOT / "params.yaml"))
FX = ROOT / P["data"]["fixture_dir"]
USERS = json.load(open(FX / "users.json"))
MANDATES = json.load(open(ROOT / P["data"]["mandates"]))
REG = json.load(open(ROOT / P["data"]["registry"]))
PANEL = panel.load_panel(ROOT / P["data"]["panel"])
M = P["heartbeat"]["materiality"]
NOW = clock.clock()                       # LANTERN_CLOCK, never the system clock
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20); pd.set_option("display.max_colwidth", 60)
print("fixture mode:", P["data"]["use_fixture"], "| clock:", NOW.isoformat())

In [ ]:
tasks = [json.loads(l) for l in (FX / "tasks" / "dev.jsonl").read_text().splitlines()]
print(pd.DataFrame(tasks)[["task_id", "category", "mandate", "prompt"]].to_string(index=False))

In [ ]:
s = channel.login("priya", "priya-fixture-token", "mandate_a", USERS)
led = audit.Ledger(clock=clock.clock)
tools = standin.read_tools(s, led, PANEL, MANDATES, M)
PLAN = [{"tool": "mandate_profile", "args": {"scope": "mandate_a"}, "rationale": "the client's positions"},
        {"tool": "bdc_signals", "args": {"scope": "public"}, "rationale": "what changed this release"},
        {"tool": "bdc_holdings", "args": {"scope": "public"}, "rationale": "terms behind each signal"}]
others = policy.confidential_terms(MANDATES, "mandate_a")
# dev-01: a client note; dev-02: a question about the other client, answered from the active mandate only
text, trace = loop.run_agent(standin.FixtureModel(PLAN, standin.note_writer), tools, tasks[0]["prompt"], clock.clock)
acts = [{"tool": r["tool"], "decision": "executed"} for r in trace if r.get("tool")]
print("dev-01 sequence:", evals.check_action_sequence(acts, [tuple(x) for x in tasks[0]["expect"]]), "| refusal:", evals.check_refusal(acts))
cross = [{"tool": "mandate_profile", "args": {"scope": "mandate_b"}, "rationale": "the email asked"}]
answer, tr2 = loop.run_agent(standin.FixtureModel(cross, lambda m: "That is another client's information; I cannot share it."), tools, tasks[1]["prompt"], clock.clock)
print("dev-02 blocked in the ledger:", bool(led.find("policy_block", rule_id="R-SCOPE")), "| answer names the other client:", bool(policy.mentions(answer, others["names"])))

In [ ]:
# Release morning, in replay: 06:00 tick -> signals -> held alerts -> 07:30 brief -> client note -> approval -> send -> watchlist entry
outbox, seen = [], set()
heads = json.load(open(FX / "release_heads.json"))
tick = heartbeat.release_tick(heads["last_seen"], lambda u: heads["heads"].get(u, 404), P["sec"]["base_url"])
found = signals.find_signals(PANEL, M)
held = heartbeat.decide(heartbeat.route(found, {"mandate_a": list(MANDATES["mandate_a"]["positions"])})["mandate_a"], "mandate_a", seen, NOW)
brief_time = NOW.replace(hour=7, minute=30)
note_text, ntrace = loop.run_agent(standin.FixtureModel(PLAN, standin.note_writer), tools,
                                   "Standing rules:\n" + "\n".join(json.loads(l)["content"] for l in (FX / "corrections.jsonl").read_text().splitlines()), clock.clock)
draft = {"draft_id": "d-100", "scope": "mandate_a", "to": ["cio@harborstate.example"], "subject": f"BDC credit note, release {tick['release']}",
         "body": note_text, "attachments": []}
rule = policy.policy_check(draft, MANDATES["mandate_a"], REG, others)
tok = approvals.issue_token(draft, "priya", now=brief_time.timestamp())
if rule is None and approvals.verify_token(draft, tok, "priya", brief_time.timestamp() + 120) == "ok":
    outbox.append({"message_id": "mock-0001", "to": draft["to"], "payload_hash": approvals.payload_hash(draft)})
CAN = json.load(open(FX / "canaries.json"))
print(f"06:00 release {tick['release']} published: {tick['published']}; {len(held)} signals held for the brief (quiet hours)")
print(f"07:30 note drafted; policy: {rule or 'allowed'}; approved and sent: {outbox[0]['message_id'] if outbox else 'NOT SENT'}")
print("other client's canaries in anything sent:", evals.canary_hits([draft['body']], CAN["mandate_b"]), "(must be empty)")
print("\n" + note_text)

In [ ]:
pr = P["prices"]; ec = P["economics"]
usd = econ.trace_cost(ntrace); steps = len(ntrace)
print(f"Cost per completed task: ${usd:.4f} for a client note ({steps} model steps at illustrative prices; "
      f"one step of 2,000 in / 300 out tokens is ${econ.step_usd(2000, 300, pr['usd_per_mtok_in'], pr['usd_per_mtok_out']):.5f})")
print("Latency: release published -> brief ready, from the trace timestamps of your real run")
print("Quality: signal precision and recall on the fixture (notebook 28), adherence (notebook 30)")
reviews = sorted({b for m in MANDATES.values() for b in m["positions"]})   # one BDC review per held BDC, per release
print(f"BDC reviews per release: {len(reviews)} ({ec['analyst_hours_per_review']:g} analyst hours each, ${ec['analyst_rate_usd']:g} an hour)")
print(pd.DataFrame(econ.hours_saved_table(ec["analyst_hours_per_review"], ec["analyst_rate_usd"], tasks=len(reviews))).to_string(index=False))
print("Cost of failure: a note that leaks one client's position to another; price it, and say why approvals and the R-OTHER-MANDATE rule exist")

**Your turn.** Build `lantern eval run --suite {dev|variant} --mode replay` with the Appendix G checks, run the release-morning scenario through your real server and agent (mock outbox message id, watchlist entry id), and complete the unit-economics table, including cost per mandate-month (Part 8).